# Cleaning log: Sephora CA SKU scrape

Every step that changes, drops, or reshapes rows is written down here with **why** and the
**row counts before and after**, so the analysis table can be traced back to the raw scrape.

How to read it:
- Each step ends with a `log_step(...)` call. It records rows and products before/after and
  keeps the dropped rows, so nothing disappears silently.
- The **Cleaning log** table at the bottom is the summary. It's also saved as a CSV next to the
  cleaned data.
- Decisions that are judgement calls (which categories to exclude, how to treat shades) live
  in clearly named constants near the step that uses them. Change them there, rerun, and the
  log updates.

Input: `data/raw/sephora_ca_2026-09-26_run1_skus.csv.gz` (one row per SKU, run 1, CAD).

In [1]:
from pathlib import Path

import pandas as pd

from shelf_life.preprocessing import parse_price, size_columns

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

RAW_PATH = Path("../data/raw/sephora_ca_2026-09-26_run1_skus.csv.gz")
OUT_DIR = Path("../data/processed")

## Logging helper

`log_step` compares the table before and after a step. Row identity is the DataFrame index,
which stays the raw row number all the way through (no `reset_index` until the end), so
"dropped" always means "these raw rows".

In [2]:
LOG: list[dict] = []
DROPPED: dict[str, pd.DataFrame] = {}


def log_step(step: str, before: pd.DataFrame, after: pd.DataFrame, why: str) -> pd.DataFrame:
    """Record one cleaning step and return the rows it removed."""
    dropped = before.loc[~before.index.isin(after.index)]
    entry = {
        "step": step,
        "rows_before": len(before),
        "rows_after": len(after),
        "rows_removed": len(before) - len(after),
        "products_before": before["product_code"].nunique(),
        "products_after": after["product_code"].nunique(),
        "why": why,
    }
    # Rerunning a cell replaces its entry instead of appending a duplicate.
    LOG[:] = [e for e in LOG if e["step"] != step]
    LOG.append(entry)
    if len(dropped):
        DROPPED[step] = dropped
    else:
        DROPPED.pop(step, None)
    print(
        f"{step}: {entry['rows_before']:,} -> {entry['rows_after']:,} rows "
        f"({entry['rows_removed']:,} removed); "
        f"{entry['products_before']:,} -> {entry['products_after']:,} products"
    )
    return dropped

## 0. Load the raw scrape

In [3]:
raw = pd.read_csv(RAW_PATH, compression="gzip")
print(raw.shape)
print(f"run_id: {raw['run_id'].unique().tolist()}, country: {raw['country'].unique().tolist()}, "
      f"currency: {raw['currency'].unique().tolist()}")
print(f"scraped between {raw['created_at'].min()} and {raw['created_at'].max()}")
_ = log_step("0. raw scrape", raw, raw, "Starting point, as exported from the scraper.")
raw.head(3)

(8742, 40)
run_id: [1], country: ['CA'], currency: ['CAD']
scraped between 2026-09-26 03:31:35 and 2026-10-07 02:35:27
0. raw scrape: 8,742 -> 8,742 rows (0 removed); 2,570 -> 2,570 products


,id,run_id,country,currency,target_url,full_product_url,product_code,loves_count,rating,reviews,brand_source_id,category_id,category_name,category_url,sku_id,brand_name,display_name,ingredients,limited_edition,first_access,limited_time_offer,new_product,online_only,few_left,out_of_stock,price,sale_price,max_purchase_quantity,size,type,url,variation_type,variation_value,returnable,finish_refinement,size_refinement,short_description,long_description,suggested_usage,created_at
0,5385,1,CA,CAD,/product/makeup-by-mario-master-mattes-tm-liquid-liner-P01018539,https://www.sephora.com/product/makeup-by-mario-master-mattes-tm-liquid-line...,P01018539,23118,3.5801,231.0,6332.0,cat60028 --- cat130054 --- cat140006,Eyeliner --- Eye --- Makeup,/shop/eyeliner --- /shop/eye-makeup --- /shop/makeup-cosmetics,2389542,MAKEUP BY MARIO,Master Mattes™ Liquid Eyeliner,"Water, Methylpropandiol, Styrene/Acrylates Copolymer, Styrene/Acrylates/Ammo...",0.0,0,0,0,0.0,0.0,0,$36.00,NaN,10,0.04 oz/ 1.2 mL,Standard,NaN,Size,0.04 oz/ 1.2 mL,1.0,Matte,NaN,"<b>What it is: </b>A pro liquid liner formula that provides long-lasting, wa...","<b>What it is: </b>A pro liquid liner formula that provides long-lasting, wa...","<b>Suggested Usage:</b><br>-In front of a mirror, rest your elbow on a flat ...",2026-10-03 23:50:05
1,8864,1,CA,CAD,/product/rare-beauty-by-selena-gomez-liquid-touch-foundation-brush-P04141879,https://www.sephora.com/product/rare-beauty-by-selena-gomez-liquid-touch-fou...,P04141879,173934,4.3504,645.0,6317.0,cat60258 --- cat60222 --- cat140006,Face Brushes --- Brushes & Applicators --- Makeup,/shop/foundation-brushes-face-brushes --- /shop/makeup-brushes --- /shop/mak...,2362408,Rare Beauty by Selena Gomez,Liquid Touch Foundation Brush,NaN,0.0,0,0,0,0.0,0.0,0,$41.00,NaN,10,NaN,Standard,NaN,NaN,NaN,1.0,NaN,NaN,"<b>What it is: </b>A cruelty-free foundation brush packed with soft, silky b...","<b>What it is: </b>A cruelty-free foundation brush packed with soft, silky b...",<b>Suggested Usage:</b><br>-Liquid Touch Foundation Brush can be used with a...,2026-10-04 04:14:52
2,2422,1,CA,CAD,/product/pro-filtr-instant-retouch-primer-P04897543,https://www.sephora.com/product/pro-filtr-instant-retouch-primer-P04897543,P04897543,167927,4.1027,2065.0,6200.0,cat60012 --- cat130058 --- cat140006,Face Primer --- Face --- Makeup,/shop/makeup-primer-face-primer --- /shop/face-makeup --- /shop/makeup-cosme...,1925528,Fenty Beauty by Rihanna,Pro Filt'r Instant Retouch Primer,"Water, Dimethicone, Isododecane, Glycerin, Cetyl Peg/Ppg-10/1 Dimethicone, P...",0.0,0,0,0,1.0,0.0,1,$49.00,$29.40,0,1.08 oz/ 32 mL,Standard,NaN,Size,1.08 oz/ 32 mL,1.0,Matte,NaN,<b>Skin type:</b><br>✔ Normal<br>✔ Dry<br>✔ Combination<br>✔ Oily<br><br><b>...,<b>Skin type:</b><br>✔ Normal<br>✔ Dry<br>✔ Combination<br>✔ Oily<br><br><b>...,<b>Suggested Usage:</b><br>-One pump is all it takes for an instant retouch ...,2026-10-03 16:20:52


## 1. Check one row per SKU

Not a filter, just a guard: if a rerun ever produces duplicate `sku_id`s, stop here rather
than double count.

In [4]:
assert raw["sku_id"].is_unique, raw[raw["sku_id"].duplicated(keep=False)]
df = raw.copy()
_ = log_step("1. check sku_id unique", raw, df, "Guard only; no duplicate sku_ids found.")

1. check sku_id unique: 8,742 -> 8,742 rows (0 removed); 2,570 -> 2,570 products


## 2. Drop rows with no price

Without a price there's nothing to analyze. These are mostly BIRTHDAY GIFT freebies (which
also have no brand or category) plus a handful of SKUs whose price didn't load.

In [5]:
before = df
df = df[df["price"].notna()]
dropped = log_step("2. drop missing price", before, df,
                   "No price -> can't be used for price analysis (incl. BIRTHDAY GIFT freebies).")
dropped[["brand_name", "display_name", "category_name", "size"]]

2. drop missing price: 8,742 -> 8,736 rows (6 removed); 2,570 -> 2,568 products


,brand_name,display_name,category_name,size
2162,NaN,BIRTHDAY GIFT Bumble and bumble Shampoo and Conditioner Set,NaN,NaN
2675,shu uemura,Ultimate Reset Strengthening Hair Mask for Very Damaged Hair,Hair Masks --- Hair Styling & Treatments --- Hair,2.5 oz/75 ml
2808,NaN,"BIRTHDAY GIFT KVD BEAUTY TATTOO LINER, SETTING POWDER, AND LIPSTICK SET",NaN,NaN
5859,LANEIGE,Water Bank Blue Hyaluronic Cream Moisturizer for Moisture Barrier Repair,Face Creams --- Moisturizers --- Skincare,MATIN KIM 1.69 fl. oz./50 ml
7577,LANEIGE,Cream Skin Milky Toner with Ceramides & Peptides for Nourishing Hydration,Toners --- Cleansers --- Skincare,MATIN KIM 2.71 fl. oz./80 ml Mist Pump
8123,L'Oréal Professionnel,"Metal Detox Sulfate Free Repair Shampoo for Dry, Damaged Hair",Shampoo --- Shampoo & Conditioner --- Hair,1/1


## 3. Fill or drop missing brand

Some SKUs lost their brand but another SKU of the same product still has it, so fill from
the product first and only drop what's still missing.

In [6]:
before = df
brand_by_product = df.dropna(subset=["brand_name"]).groupby("product_code")["brand_name"].first()
missing = df["brand_name"].isna()
df = df.assign(brand_name=df["brand_name"].fillna(df["product_code"].map(brand_by_product)))
print(f"{missing.sum()} rows missing brand, "
      f"{(missing & df['brand_name'].notna()).sum()} filled from same product_code")
df = df[df["brand_name"].notna()]
dropped = log_step("3. fill/drop missing brand", before, df,
                   "Brand filled from other SKUs of the same product; rows still blank dropped.")
before.loc[missing, ["product_code", "display_name", "brand_name"]]

2 rows missing brand, 0 filled from same product_code
3. fill/drop missing brand: 8,736 -> 8,734 rows (2 removed); 2,568 -> 2,567 products


,product_code,display_name,brand_name
2900,P443930,"Soleil Glow Bronzer for Buildable, Luminous Finish",NaN
2901,P443930,"Soleil Glow Bronzer for Buildable, Luminous Finish",NaN


## 4. Parse prices (CAD)

`"$36.00"` -> `36.0`. `on_sale` marks the ~100 SKUs with a `sale_price`. Prices are list
prices on sephora.com/ca, so CAD.

In [7]:
before = df
df = df.assign(
    price_cad=df["price"].map(parse_price),
    sale_price_cad=df["sale_price"].map(parse_price),
)
df["on_sale"] = df["sale_price_cad"].notna()
assert df["price_cad"].notna().all()
_ = log_step("4. parse prices", before, df, "Text prices -> numeric CAD; no rows removed.")
df[["price", "price_cad", "sale_price", "sale_price_cad"]].describe()

4. parse prices: 8,734 -> 8,734 rows (0 removed); 2,567 -> 2,567 products


,price_cad,sale_price_cad
count,8734.000000,102.000000
mean,66.204473,46.715392
std,65.013490,34.793875
min,6.600000,8.750000
25%,36.000000,20.437500
50%,49.000000,38.565000
75%,68.000000,63.847500
max,1755.000000,165.200000


## 5. Split categories

`category_name` is `"leaf --- group --- top"` (2 or 3 levels). Split it into
`category` (top), `group` (middle, blank for 2-level paths) and `subcategory` (leaf).

Note: **Mini Size** is a top-level category on Sephora as well as a leaf inside Makeup,
Skincare, etc. Those rows are kept and flagged with `is_mini` so you can include or exclude
minis in a given analysis.

In [8]:
before = df
parts = df["category_name"].str.split(" --- ")
df = df.assign(
    category=parts.str[-1],
    group=parts.map(lambda p: p[1] if len(p) == 3 else pd.NA),
    subcategory=parts.str[0],
)
df["is_mini"] = (df["category"] == "Mini Size") | (df["subcategory"] == "Mini Size")
_ = log_step("5. split categories", before, df, "category/group/subcategory columns; no rows removed.")
df["category"].value_counts()

5. split categories: 8,734 -> 8,734 rows (0 removed); 2,567 -> 2,567 products


category
Makeup         5860
Fragrance       934
Skincare        915
Hair            735
Bath & Body     193
Mini Size        97
Name: count, dtype: int64

## 6. Fix URLs and drop unusable columns

- `full_product_url` comes back US-style (`sephora.com/product/...`). Rebuild it as the
  Canadian URL from `target_url` so links match the prices.
- `url` is empty for every row: dropped.
- `few_left` is set on ~40% of SKUs, which doesn't match what the site shows, so it's dropped
  rather than risk someone using it as a stock signal.

In [9]:
before = df
print("url non-null:", df["url"].notna().sum())
print(f"few_left share: {df['few_left'].mean():.0%}")
df = df.assign(product_url_ca="https://www.sephora.com/ca/en" + df["target_url"])
df = df.drop(columns=["url", "few_left", "full_product_url"])
_ = log_step("6. CA urls, drop url/few_left", before, df,
         "US-style URLs rebuilt as /ca/en; empty url and unreliable few_left dropped.")
df["product_url_ca"].head(3).to_list()

url non-null: 0
few_left share: 42%
6. CA urls, drop url/few_left: 8,734 -> 8,734 rows (0 removed); 2,567 -> 2,567 products


['https://www.sephora.com/ca/en/product/makeup-by-mario-master-mattes-tm-liquid-liner-P01018539',
 'https://www.sephora.com/ca/en/product/rare-beauty-by-selena-gomez-liquid-touch-foundation-brush-P04141879',
 'https://www.sephora.com/ca/en/product/pro-filtr-instant-retouch-primer-P04897543']

## 7. Exclude categories I can't measure

The analysis is about price per amount of product, so anything sold as a tool or object
(hair curlers, brushes, pillowcases...) is out.

**`EXCLUDED_SUBCATEGORIES` is the list to maintain.** Keys are `(category, subcategory)`
so a leaf like "Mini Size" can be kept in one category and dropped in another. Add a reason
for each, it ends up in the dropped-rows file.

> TODO (Hannah): only Curling Irons is confirmed so far. The table below the next cell lists
> every subcategory where *no* SKU has a usable size, as candidates. Move the ones you want
> excluded into the dict.

In [10]:
EXCLUDED_SUBCATEGORIES: dict[tuple[str, str], str] = {
    ("Hair", "Curling Irons"): "hair tool, no product amount",
}

before = df
excluded_key = pd.Series(list(zip(df["category"], df["subcategory"])), index=df.index)
is_excluded = excluded_key.isin(EXCLUDED_SUBCATEGORIES.keys())
df = df[~is_excluded]
dropped = log_step("7. exclude unmeasurable categories", before, df,
                   f"{len(EXCLUDED_SUBCATEGORIES)} subcategories excluded; see EXCLUDED_SUBCATEGORIES.")
dropped.assign(reason=excluded_key[is_excluded].map(EXCLUDED_SUBCATEGORIES)).groupby(
    ["category", "subcategory", "reason"]).size().rename("rows")

7. exclude unmeasurable categories: 8,734 -> 8,730 rows (4 removed); 2,567 -> 2,563 products


category  subcategory    reason                      
Hair      Curling Irons  hair tool, no product amount    4
Name: rows, dtype: int64

## 8. Parse sizes

`"1.7 oz/ 50 mL"` -> numeric `size_oz`, `size_fl_oz`, `size_ml`, `size_g` via
`shelf_life.preprocessing.size_columns`.

One extra fix here: ~400 sizes have no units at all, like `"1.7/50"`. When the second number
is 25-35x the first, it's the usual "oz / mL-or-g" pair with the units missing, so the first
number is taken as `size_oz`. Pairs like `"1/1"` stay unusable.

In [11]:
OZ_PAIR_RATIO = (25, 35)  # 1 fl oz = 29.6 mL, 1 oz = 28.3 g


def unitless_oz(size) -> float | None:
    """'1.7/50' -> 1.7 when it looks like an oz / (mL or g) pair, else None."""
    if not isinstance(size, str):
        return None
    parts = [p.strip() for p in size.split("/")]
    try:
        first, second = (float(p) for p in parts)
    except ValueError:
        return None
    if first > 0 and OZ_PAIR_RATIO[0] <= second / first <= OZ_PAIR_RATIO[1]:
        return first
    return None


before = df
sizes = pd.DataFrame(df["size"].map(size_columns).to_list(), index=df.index)
df = pd.concat([df, sizes], axis=1)

inferred = df["size"].map(unitless_oz)
fill = df[["size_oz", "size_fl_oz", "size_ml", "size_g"]].isna().all(axis=1) & inferred.notna()
df.loc[fill, "size_oz"] = inferred[fill]
df["size_units_inferred"] = fill

# One "oz" number per SKU for price_per_oz: weight oz, else fluid oz.
df["amount_oz"] = df["size_oz"].fillna(df["size_fl_oz"])
_ = log_step("8. parse sizes", before, df, "Size text -> numeric columns; unitless oz pairs inferred.")

outcome = pd.Series("parsed", index=df.index)
outcome[df["size"].isna()] = "no size text"
outcome[df["size"].notna() & df["amount_oz"].isna() & df["size_ml"].isna()
        & df["size_g"].isna()] = "unparseable"
outcome[fill] = "units inferred"
outcome.value_counts()

8. parse sizes: 8,730 -> 8,730 rows (0 removed); 2,563 -> 2,563 products


parsed            8196
no size text       301
units inferred     160
unparseable         73
Name: count, dtype: int64

In [12]:
# What the unparseable ones look like
df.loc[outcome == "unparseable", "size"].value_counts().head(20)

size
1/1                                                    11
5 Treatments                                            3
6 pack                                                  3
60 Treatments                                           2
30 Treatments + 5 Bonus                                 2
0.001 / 0.04                                            2
Creme .05 OZ Powder .09 OZ/Creme 1.6 G Powder 2.6 G     2
0.017                                                   2
60 pads/pack                                            1
90 pads                                                 1
28 pads/pack                                            1
45 pads                                                 1
6 count/6 count                                         1
10 masks / pack                                         1
4 Hair Clips                                            1
2.48'' D x 2.48'' H x 10.24'' W                         1
5w x 5.3d x 3.97h\t\t\t\t\t\t                           1
30 treatm

## 9. Drop rows without a usable size

No amount -> no unit price. Shown by category first, because if a whole subcategory lands
here it's probably a candidate for the exclusion list in step 7 instead.

In [13]:
has_size = df[["amount_oz", "size_ml", "size_g"]].notna().any(axis=1)

# Candidates for EXCLUDED_SUBCATEGORIES: subcategories where no SKU has a usable size.
by_sub = df.assign(has_size=has_size).groupby(["category", "subcategory"])["has_size"].agg(
    rows="size", with_size="sum")
print("Subcategories with no usable size at all:")
display(by_sub[by_sub["with_size"] == 0])

Subcategories with no usable size at all:


rows  with_size
category    subcategory                                       
Bath & Body Value & Gift Sets                     1          0
Fragrance   Candles & Home Scents                 1          0
Hair        Brushes & Combs                       5          0
            Hair Clips & Claw Clips               1          0
            Hair Dryers                           2          0
            Hair Tool Attachments & Diffusers     1          0
            Hair Towels                           1          0
            Scalp Massagers & Rollers             1          0
            Silk Pillowcases                     17          0
            Value & Gift Sets                     1          0
Makeup      Eye Brushes                           6          0
            Eyelash Curlers                       4          0
            Lip Brushes                           1          0
            Makeup Palettes                       2          0
            Sharpeners                            2          0
            Sponges & Applicators                 7          0
Skincare    Anti-Aging                            3          0
            Eye Masks                             2          0
            Facial Rollers                        1          0
            Holistic Wellness                     6          0
            Sheet Masks                           3          0
            Value & Gift Sets                     2          0

In [14]:
before = df
df = df[has_size]
dropped = log_step("9. drop rows without usable size", before, df,
                   "No numeric amount -> unit price impossible.")
dropped.groupby("category").size().rename("rows dropped")

9. drop rows without usable size: 8,730 -> 8,356 rows (374 removed); 2,563 -> 2,435 products


category
Bath & Body      1
Fragrance       10
Hair            46
Makeup         255
Mini Size        3
Skincare        59
Name: rows dropped, dtype: int64

## 10. Collapse shade variants

5,878 SKUs are colour variants (foundation shades, lipstick colours). Same product, same size,
same price, so for price analysis they'd let a 40-shade foundation count 40 times and make
Makeup look like two thirds of the store.

Collapse to **one row per product, parsed size and price** (parsed, so `"0.002 oz/ 0.06 g"`
and `"0.002 oz/0.06 g"` count as the same size), keeping `n_shades` so the information
isn't lost. Products whose shades differ in price or size keep one row per distinct
price/size.

In [15]:
before = df
key = ["product_code", "amount_oz", "size_ml", "size_g", "price_cad"]
n_variants = df.groupby(key, dropna=False)["sku_id"].transform("size")
df = df.assign(n_variants=n_variants).drop_duplicates(subset=key, keep="first")
_ = log_step("10. collapse shade variants", before, df,
         "One row per product_code + parsed size + price; n_variants keeps the count.")

share = pd.DataFrame({
    "before": before["category"].value_counts(normalize=True),
    "after": df["category"].value_counts(normalize=True),
}).round(3)
share

10. collapse shade variants: 8,356 -> 3,539 rows (4,817 removed); 2,435 -> 2,435 products


,before,after
category,,
Makeup,0.671,0.273
Fragrance,0.111,0.261
Skincare,0.102,0.221
Hair,0.082,0.191
Bath & Body,0.023,0.041
Mini Size,0.011,0.013


## 11. Unit prices

`price_per_oz` uses `amount_oz` (weight or fluid oz). `price_per_ml` and `price_per_g` are
there for when you want to stay within liquids or solids.

In [16]:
before = df
df = df.assign(
    price_per_oz=df["price_cad"] / df["amount_oz"],
    price_per_ml=df["price_cad"] / df["size_ml"],
    price_per_g=df["price_cad"] / df["size_g"],
)
_ = log_step("11. unit prices", before, df, "price_cad / amount; no rows removed.")
df.groupby("category")["price_per_oz"].describe().round(2)

11. unit prices: 3,539 -> 3,539 rows (0 removed); 2,435 -> 2,435 products


,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
Bath & Body,145.0,15.71,27.30,2.00,6.33,9.10,16.31,295.65
Fragrance,914.0,106.92,61.90,5.68,65.12,93.53,137.76,436.36
Hair,673.0,13.07,34.10,1.95,5.41,7.60,11.50,714.29
Makeup,927.0,874.64,5621.53,0.81,106.70,200.00,396.90,155000.00
Mini Size,44.0,356.89,609.43,7.92,11.67,70.59,417.50,2750.00
Skincare,766.0,82.78,127.86,1.98,15.02,43.52,94.78,1766.67


## Cleaning log

The summary of every step above. Saved with the cleaned data, along with every dropped row
and the step that dropped it.

In [17]:
log = pd.DataFrame(LOG)
log

,step,rows_before,rows_after,rows_removed,products_before,products_after,why
0,0. raw scrape,8742,8742,0,2570,2570,"Starting point, as exported from the scraper."
1,1. check sku_id unique,8742,8742,0,2570,2570,Guard only; no duplicate sku_ids found.
2,2. drop missing price,8742,8736,6,2570,2568,No price -> can't be used for price analysis (incl. BIRTHDAY GIFT freebies).
3,3. fill/drop missing brand,8736,8734,2,2568,2567,Brand filled from other SKUs of the same product; rows still blank dropped.
4,4. parse prices,8734,8734,0,2567,2567,Text prices -> numeric CAD; no rows removed.
5,5. split categories,8734,8734,0,2567,2567,category/group/subcategory columns; no rows removed.
6,"6. CA urls, drop url/few_left",8734,8734,0,2567,2567,US-style URLs rebuilt as /ca/en; empty url and unreliable few_left dropped.
7,7. exclude unmeasurable categories,8734,8730,4,2567,2563,1 subcategories excluded; see EXCLUDED_SUBCATEGORIES.
8,8. parse sizes,8730,8730,0,2563,2563,Size text -> numeric columns; unitless oz pairs inferred.
9,9. drop rows without usable size,8730,8356,374,2563,2435,No numeric amount -> unit price impossible.


In [18]:
COLUMNS = [
    "sku_id", "product_code", "brand_name", "display_name",
    "category", "group", "subcategory", "is_mini",
    "variation_type", "variation_value", "n_variants",
    "size", "size_oz", "size_fl_oz", "size_ml", "size_g", "size_units_inferred", "amount_oz",
    "price_cad", "sale_price_cad", "on_sale", "price_per_oz", "price_per_ml", "price_per_g",
    "rating", "reviews", "loves_count",
    "limited_edition", "new_product", "online_only", "out_of_stock",
    "product_url_ca", "created_at",
]
clean = df[COLUMNS].reset_index(names="raw_row")

dropped_rows = pd.concat(
    [d.assign(dropped_at=step) for step, d in DROPPED.items()]
).reset_index(names="raw_row")[["raw_row", "dropped_at", "sku_id", "product_code",
                                  "brand_name", "display_name", "category_name", "size", "price"]]

OUT_DIR.mkdir(parents=True, exist_ok=True)
stem = RAW_PATH.name.removesuffix(".csv.gz")
clean.to_csv(OUT_DIR / f"{stem}_clean.csv", index=False)
log.to_csv(OUT_DIR / f"{stem}_cleaning_log.csv", index=False)
dropped_rows.to_csv(OUT_DIR / f"{stem}_dropped_rows.csv", index=False)
print(f"clean: {clean.shape}, dropped: {len(dropped_rows):,} rows -> {OUT_DIR}")

clean: (3539, 34), dropped: 5,203 rows -> ../data/processed


## Known issues and open questions

Not handled yet, written down so they don't get forgotten:

1. **Exclusion list** (step 7) only has Curling Irons. The candidates table in step 9 lists
   subcategories with no usable size; most of those are dropped by step 9 anyway, but listing
   them in step 7 makes the reason explicit.
2. **Multipacks** like `"3 x 0.002 oz (3-pack)"`: the `3 x` is recorded in `size_info` as `x3`
   but not multiplied into the amount, so price per oz is 3x too high for those.
3. **Tiny pencils** (brow, liner) have amounts like 0.0002 oz, which gives price_per_oz in the
   tens of thousands. They're real numbers, but they'll dominate any Makeup average, so use
   medians or compare within subcategory.
4. **Minis** are kept and flagged `is_mini`; decide per analysis whether they belong.
5. **Sizes with no unit** that aren't oz pairs (`"1/1"`, `"2/5"`) are dropped in step 9. A few
   might be recoverable from `variation_value` or `size_refinement`.